In [ ]:
import torch
import torch.nn as nn
import math


In [ ]:
#Faster GLU approximation
def _gelu(x : torch.Tensor) -> torch.Tensor:
    return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

In [ ]:
class ExpertFFN(nn.Module):
  def __init__(self, d_model : int, hidden_dim : int, dropout_rate : float = 0.0):
      """
        A 2-layer MLP expert. Hidden dim is usually smaller than a dense FFN(e.g., 0.25 × d_model in DeepSeek-V3).
      """
      super().__init__()

      self.d_model = d_model
      self.hidden_dim = hidden_dim
      self.dropout_rate = dropout_rate

      self.fc1 = nn.Linear(d_model, hidden_dim) #Expansion
      self.fc2 = nn.Linear(hidden_dim, d_model) #Contraction
      self.dropout = nn.Dropout(dropout_rate)
      self._gelu = _gelu

  def forward(self, x : torch.Tensor) -> torch.Tensor:
      x = self._gelu(self.fc1(x))
      x = self.fc2(x)
      x = self.dropout(x) #Dropout applied after contraction expansion phase

      return x

In [ ]:
x = torch.rand(3, 5, 16)
effn = ExpertFFN(16, 64, 0.1)
effn(x).shape

torch.Size([3, 5, 16])

#### Design Question: How to pass expert selection matrix to auxilliary loss function from each transformer block

#Design Question:

In [ ]:
class DeepSeekMoE(nn.Module):
    def __init__(
                  self,
                  num_experts : int,
                  d_model : int, #embedding dim,
                  num_routed_exp : int,
                  num_shared_exp : int,
                  top_k : int, #maximal number of activated experts
                  routed_hidden_dim : int , #Expert FFN hidden dim
                  shared_hidden_dim : int, #Expert FFN hidden dim
                  dropout_rate : float,
                  bias_lr : float = 0.0, #scaling parameter of auxilliary loss or load balancing loss
    ):
        super().__init__()
        assert top_k <= num_experts

        self.num_experts = num_experts
        self.num_routed_exp = num_routed_exp
        self.num_shared_exp = num_shared_exp

        self.d_model = d_model
        self.routed_hidden_dim = routed_hidden_dim
        self.shared_hidden_dim = shared_hidden_dim
        self.top_k = top_k
        self.dropout_rate = dropout_rate
        self.bias_lr = bias_lr

        self.routed_experts = nn.ModuleList([ExpertFFN(d_model, routed_hidden_dim, dropout_rate) for _ in range(self.num_routed_exp)])
        shared_hidden_dim = routed_hidden_dim if self.num_shared_exp == 0 else shared_hidden_dim
        self.shared_experts = nn.ModuleList([ExpertFFN(d_model, shared_hidden_dim, dropout_rate) for _ in range(self.num_shared_exp)])
        #self.router_layer = nn.Linear(d_model, num_experts)

        #Registering directly Expert Selector Weight Matrix
        self.register_parameter("centroids",nn.Parameter(torch.empty(self.num_routed_exp, d_model)))
        nn.init.normal_(self.centroids, std=d_model ** -0.5)

        self.register_parameter("bias", nn.Parameter(torch.zeros(self.num_routed_exp)))


    def forward(self, x):
        out = None

        batch_size, seq_len, emb_dim = x.shape
        x_flat = x.reshape(-1, emb_dim)

        out_shared_exp = torch.zeros_like(x)
        out_routed_exp = torch.zeros_like(x_flat)

        #Shared Experts Path
        for exp in self.shared_experts:
            out_shared_exp += exp(x)

        #The Routed Expert Path
        use_autocast = self.fp16_router and x.is_cuda
        device_type = "cuda" if x.is_cuda else x.device.type
        with torch.autocast(device_type = device_type, enabled=use_autocast):
            logits = F.linear(x_flat, self.centroids)
            logits = logits + self.bias.to(logits.dtype)

        topk_logits, topk_indices = torch.topk(logits, k = self.top_k, dim = -1)
        gate = F.softmax(topk_logits, dim = -1)


        for i in range(self.num_routed_exp):
            mask = topk_indices == i
            row_idx, col_idx = mask.nonzero(as_tuple = True)
            if row_idx.nelement() == 0:
                continue

            exp_in = x_flat.index_select(0, row_idx)
            routed_out = self.routed_experts[i](exp_in)
            w = gate[row_idx, col_idx].unsqueeze(-1)
            routed_out = routed_out * w
            out_routed_exp.index_add_(0, row_idx, routed_out)


        out_routed_exp = out_routed_exp.reshape(batch_size, seq_len, emb_dim)
        out = x + out_shared_exp + out_routed_exp

        return out

    @torch.no_grad()
    def update_batch(self, x : torch.Tensor):
        batch_size, seq_len, emb_dim = x.shape
        x_flat = x.reshape(-1, emb_dim)
        eps = 1e-6

        logits = F.linear(x_flat, self.centroids)
        logits = logits + self.bias.to(logits.dtype)
        topk_logits, topk_indices = torch.topk(logits, k = self.top_k, dim = -1)
        expert_counts = torch.bincount(topk_indices.flatten(), minlength = self.num_experts).float()
        avg = torch.sum(expert_counts) / max(1, self.num_routd_exp)
        violation = (counts - avg) / (avg + eps)
        self.bias_lr.add(self.bias_lr * torch.tanh(violation))


In [ ]:
x = torch.rand(3, 5, 16)

In [ ]:
x_flat = x.reshape(-1, x.shape[-1])

In [ ]:
topk_logits, topk_indices = torch.topk(x_flat, k = 8, dim = -1)

In [ ]:
mask = topk_indices == 7

In [ ]:
mask.nonzero(as_tuple = True)

(tensor([ 1,  2,  4,  5,  7, 10, 11, 14]), tensor([1, 7, 4, 2, 1, 3, 0, 5]))

In [ ]:
row_idx, col_idx = mask.nonzero(as_tuple = True)

In [ ]:
col_idx

tensor([1, 7, 4, 2, 1, 3, 0, 5])

In [ ]:
x_flat.index_select(0, row_idx)

tensor([[0.2717, 0.4745, 0.2637, 0.0511, 0.3785, 0.2204, 0.0014, 0.9082, 0.9067,
         0.9508, 0.3527, 0.3173, 0.2719, 0.2220, 0.2158, 0.8895],
        [0.0561, 0.6794, 0.4253, 0.4121, 0.5941, 0.6593, 0.9911, 0.6326, 0.8848,
         0.4137, 0.4115, 0.6097, 0.7125, 0.9198, 0.1871, 0.9566],
        [0.7388, 0.2576, 0.5157, 0.0549, 0.0515, 0.4419, 0.8882, 0.8692, 0.4494,
         0.0757, 0.9114, 0.9838, 0.6476, 0.8758, 0.7213, 0.8035],
        [0.5969, 0.8410, 0.5886, 0.3801, 0.3344, 0.1961, 0.8207, 0.8462, 0.0140,
         0.6914, 0.5598, 0.9312, 0.5285, 0.5099, 0.4064, 0.9710],
        [0.9188, 0.3180, 0.4141, 0.1841, 0.9730, 0.0138, 0.2046, 0.9420, 0.6232,
         0.2980, 0.1874, 0.7307, 0.7069, 0.8754, 0.7444, 0.2974],
        [0.8155, 0.0375, 0.7954, 0.7879, 0.9529, 0.0856, 0.8733, 0.8016, 0.6484,
         0.3939, 0.1053, 0.3515, 0.1847, 0.4167, 0.0825, 0.5843],
        [0.4669, 0.8898, 0.4833, 0.5853, 0.3861, 0.0820, 0.4185, 0.8948, 0.3476,
         0.5587, 0.2272, 0.7838, 0.37